#  Business Understanding

The Sales 2015 dataset contains transactional records of customer purchases made during the year 2015.

This is the primary fact table of the project and records information about orders, products, customers, sales territories, and financial metrics.

The dataset will be used to calculate key business KPIs such as:

- Total Revenue
- Total Profit
- Sales Growth
- Customer Lifetime Value (LTV)
- Product Performance
- Regional Sales Performance
- Monthly Sales Trends

This table will later be connected to all dimension tables to build a Star Schema for SQL analysis and Power BI dashboards.

# 1. Load the Dataset

In [32]:
import pandas as pd 
import numpy as np

sales = pd.read_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/raw/AdventureWorks_Sales_2015.csv")
customers = pd.read_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/cleaned/customers_clean.csv")
products = pd.read_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/cleaned/products_clean.csv")
territories = pd.read_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/cleaned/territories_clean.csv")
returns = pd.read_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/cleaned/returns_clean.csv")


# 2. Data Profiling

In [33]:
sales.head()
sales.tail()

,OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2625,12/31/2015,11/29/2002,SO48728,354,13111,9,1,1
2626,12/31/2015,11/14/2002,SO48729,324,26563,9,1,1
2627,12/31/2015,12/2/2002,SO48724,340,20722,8,1,1
2628,12/31/2015,10/9/2002,SO48723,369,14944,7,1,1
2629,12/31/2015,11/22/2002,SO48726,383,24915,9,1,1


In [34]:
sales.shape

(2630, 8)

In [35]:
sales.columns

Index(['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey',
       'TerritoryKey', 'OrderLineItem', 'OrderQuantity'],
      dtype='object')

In [36]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2630 entries, 0 to 2629
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   OrderDate      2630 non-null   object
 1   StockDate      2630 non-null   object
 2   OrderNumber    2630 non-null   object
 3   ProductKey     2630 non-null   int64 
 4   CustomerKey    2630 non-null   int64 
 5   TerritoryKey   2630 non-null   int64 
 6   OrderLineItem  2630 non-null   int64 
 7   OrderQuantity  2630 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 164.5+ KB


In [37]:
sales.describe()

,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
count,2630.000000,2630.000000,2630.000000,2630.0,2630.0
mean,344.180608,18344.052471,6.587833,1.0,1.0
std,26.288429,5567.047291,2.915261,0.0,0.0
min,310.000000,11090.000000,1.000000,1.0,1.0
25%,314.000000,13118.250000,4.000000,1.0,1.0
50%,348.000000,16828.500000,8.000000,1.0,1.0
75%,370.000000,22859.250000,9.000000,1.0,1.0
max,389.000000,29481.000000,10.000000,1.0,1.0


In [38]:
sales.isnull().sum()

OrderDate        0
StockDate        0
OrderNumber      0
ProductKey       0
CustomerKey      0
TerritoryKey     0
OrderLineItem    0
OrderQuantity    0
dtype: int64

In [39]:
sales.duplicated().sum()

np.int64(0)

In [40]:
sales.nunique()

OrderDate         365
StockDate         446
OrderNumber      2630
ProductKey         44
CustomerKey      2630
TerritoryKey        8
OrderLineItem       1
OrderQuantity       1
dtype: int64

# 3. Data Quality Assessment

- No missing values found.
- No duplicate records found.
- Data types were validated and corrected.
- Date columns were converted to datetime format.
- Business rules were validated successfully.
- Foreign key relationships with Customers, Products, and Territories were verified.

**Conclusion:** The Sales dataset passed all data quality checks and is ready for feature engineering and loading into the data warehouse.

# 4. Data Cleaning

In [41]:
sales.columns = (
    sales.columns
    .str.replace(" ","_")
    .str.replace(r'(?<!^)(?=[A-Z])',"_",regex=True)
    .str.lower()
)

In [42]:
sales.columns

Index(['order_date', 'stock_date', 'order_number', 'product_key',
       'customer_key', 'territory_key', 'order_line_item', 'order_quantity'],
      dtype='object')

In [43]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2630 entries, 0 to 2629
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   order_date       2630 non-null   object
 1   stock_date       2630 non-null   object
 2   order_number     2630 non-null   object
 3   product_key      2630 non-null   int64 
 4   customer_key     2630 non-null   int64 
 5   territory_key    2630 non-null   int64 
 6   order_line_item  2630 non-null   int64 
 7   order_quantity   2630 non-null   int64 
dtypes: int64(5), object(3)
memory usage: 164.5+ KB


In [44]:
date_columns = ['order_date','stock_date']

for col in date_columns:
    sales[col] = pd.to_datetime(sales[col],errors='coerce')

In [45]:
sales[sales['order_date'].isna()]

,order_date,stock_date,order_number,product_key,customer_key,territory_key,order_line_item,order_quantity


In [46]:
sales.isnull().sum()

order_date         0
stock_date         0
order_number       0
product_key        0
customer_key       0
territory_key      0
order_line_item    0
order_quantity     0
dtype: int64

In [47]:
sales['order_quantity'].describe()

count    2630.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: order_quantity, dtype: float64

In [48]:
sales[
    sales["stock_date"] > sales["order_date"]
]

,order_date,stock_date,order_number,product_key,customer_key,territory_key,order_line_item,order_quantity


In [49]:
sales.columns

Index(['order_date', 'stock_date', 'order_number', 'product_key',
       'customer_key', 'territory_key', 'order_line_item', 'order_quantity'],
      dtype='object')

In [50]:
sales['customer_key'].isin(customers['customer_key']).all()

np.True_

In [51]:
sales['product_key'].isin(products['product_key']).all()

np.True_

In [52]:
sales['territory_key'].isin(territories['sales_territory_key']).all()

np.True_

In [53]:
sales = sales.merge(
    products[['product_key',"product_price",'product_cost']],
    on='product_key',
    how='left'
)

In [54]:
sales.isnull().sum()

order_date         0
stock_date         0
order_number       0
product_key        0
customer_key       0
territory_key      0
order_line_item    0
order_quantity     0
product_price      0
product_cost       0
dtype: int64

In [55]:
sales.dtypes

order_date         datetime64[ns]
stock_date         datetime64[ns]
order_number               object
product_key                 int64
customer_key                int64
territory_key               int64
order_line_item             int64
order_quantity              int64
product_price             float64
product_cost              float64
dtype: object

In [ ]:
sales['revenue'] = sales['product_price'] * sales['order_quantity']

sales['profit'] = (
    (sales['product_price'] - sales['product_cost'])
    * sales['order_quantity']
)


In [57]:
sales['profit_margin'] = (
    sales['profit'] / sales['revenue']
)*100

# 5. Export Clean Data

In [58]:
sales.to_csv("C:/Users/singh/OneDrive/Desktop/AdventureWorks_Analytics/data/cleaned/sales_2015_clean.csv"
             ,index=False)